In [ ]:
# Neuralese CVE Classification - Kaggle P100 Training
# Seed=137, GPU required, torch 2.2.0+cu118

import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '137'

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import math
import json
import pickle

# Seed everything
torch.manual_seed(137)
np.random.seed(137)
random.seed(137)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# GPU guard - REQUIRED
assert torch.cuda.is_available(), "CUDA not available - GPU REQUIRED"
device = torch.device('cuda')
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Torch: {torch.__version__}")


In [ ]:
# Neuralese Architecture (from prototype)

class NeuraleseEncoder(nn.Module):
    def __init__(self, vocab_size=32000, latent_dim=1024, max_seq_len=512):
        super().__init__()
        self.latent_dim = latent_dim
        self.token_emb = nn.Embedding(vocab_size, latent_dim)
        self.pos_emb = nn.Parameter(torch.randn(max_seq_len, latent_dim) * 0.02)
        self.proj = nn.Sequential(
            nn.Linear(latent_dim, latent_dim * 4),
            nn.GELU(),
            nn.Linear(latent_dim * 4, latent_dim),
            nn.LayerNorm(latent_dim)
        )
        
    def forward(self, input_ids):
        seq_len = input_ids.size(1)
        x = self.token_emb(input_ids) + self.pos_emb[:seq_len]
        return self.proj(x)


class NeuraleseDecoder(nn.Module):
    def __init__(self, vocab_size=32000, latent_dim=1024):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(latent_dim, latent_dim * 4),
            nn.GELU(),
            nn.Linear(latent_dim * 4, vocab_size)
        )
        
    def forward(self, latent):
        return self.proj(latent)


class LatentReasoningBlock(nn.Module):
    def __init__(self, latent_dim=1024, num_heads=16, ff_mult=4, dropout=0.1):
        super().__init__()
        self.latent_dim = latent_dim
        self.num_heads = num_heads
        self.head_dim = latent_dim // num_heads
        
        self.q_proj = nn.Linear(latent_dim, latent_dim, bias=False)
        self.k_proj = nn.Linear(latent_dim, latent_dim, bias=False)
        self.v_proj = nn.Linear(latent_dim, latent_dim, bias=False)
        self.out_proj = nn.Linear(latent_dim, latent_dim)
        
        self.ff = nn.Sequential(
            nn.Linear(latent_dim, latent_dim * ff_mult),
            nn.GELU(),
            nn.Linear(latent_dim * ff_mult, latent_dim)
        )
        
        self.norm1 = nn.LayerNorm(latent_dim)
        self.norm2 = nn.LayerNorm(latent_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        residual = x
        x = self.norm1(x)
        
        q = self.q_proj(x).view(x.size(0), x.size(1), self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(x.size(0), x.size(1), self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(x.size(0), x.size(1), self.num_heads, self.head_dim).transpose(1, 2)
        
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_output = torch.matmul(attn_weights, v)
        
        attn_output = attn_output.transpose(1, 2).contiguous().view(x.size(0), x.size(1), -1)
        x = residual + self.dropout(self.out_proj(attn_output))
        
        residual = x
        x = self.norm2(x)
        x = residual + self.dropout(self.ff(x))
        
        return x


class AdaptiveComputationTime(nn.Module):
    def __init__(self, latent_dim=1024, max_steps=8, threshold=0.99):
        super().__init__()
        self.halt = nn.Linear(latent_dim, 1)
        self.max_steps = max_steps
        self.threshold = threshold
        
    def forward(self, x, reasoning_blocks):
        batch, seq_len, _ = x.shape
        halt_probs = torch.zeros(batch, seq_len, device=x.device)
        previous_h = torch.zeros_like(x)
        n_updates = torch.zeros(batch, seq_len, device=x.device)
        
        for step in range(self.max_steps):
            h = reasoning_blocks[step % len(reasoning_blocks)](x)
            p = torch.sigmoid(self.halt(h)).squeeze(-1)
            
            halt_probs = halt_probs + p
            still_running = (halt_probs < self.threshold).float()
            
            weight = p * still_running
            previous_h = previous_h + weight.unsqueeze(-1) * h
            n_updates = n_updates + still_running
            
            if (halt_probs >= self.threshold).all():
                break
                
        return previous_h, n_updates.mean()


class NeuraleseReasoner(nn.Module):
    def __init__(self, vocab_size=32000, latent_dim=1024, num_blocks=4, max_act_steps=8, num_classes=10):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = NeuraleseEncoder(vocab_size, latent_dim)
        self.decoder = NeuraleseDecoder(vocab_size, latent_dim)
        
        self.reasoning_blocks = nn.ModuleList([
            LatentReasoningBlock(latent_dim) for _ in range(num_blocks)
])
        
        self.act = AdaptiveComputationTime(latent_dim, max_act_steps)
        self.contrastive_proj = nn.Linear(latent_dim, latent_dim)
        
        # Classification head for CVE
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, latent_dim // 2),
            nn.GELU(),
            nn.Linear(latent_dim // 2, num_classes)
        )
        
    def forward(self, input_ids, targets=None, labels=None):
        latent = self.encoder(input_ids)
        reasoned_latent, avg_steps = self.act(latent, self.reasoning_blocks)
        logits = self.decoder(reasoned_latent)
        contrastive_latent = F.normalize(self.contrastive_proj(reasoned_latent), dim=-1)
        
        # Classification from [CLS] token (first position)
        cls_latent = reasoned_latent[:, 0]
        class_logits = self.classifier(cls_latent)
        
        outputs = {
            'logits': logits,
            'class_logits': class_logits,
            'reasoned_latent': reasoned_latent,
            'contrastive_latent': contrastive_latent,
            'avg_reasoning_steps': avg_steps
        }
        
        if targets is not None and labels is not None:
            # Language modeling loss
            lm_loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            
            # Classification loss
            cls_loss = F.cross_entropy(class_logits, labels)
            
            # Contrastive loss
            with torch.no_grad():
                encoder_latent = F.normalize(self.contrastive_proj(latent), dim=-1)
            
            sim = torch.matmul(contrastive_latent, encoder_latent.transpose(-2, -1)) / 0.07
            batch, seq_len, _ = sim.shape
            sim_flat = sim.reshape(batch * seq_len, seq_len)
            contrastive_labels = torch.arange(seq_len, device=sim.device).unsqueeze(0).expand(batch, seq_len).reshape(batch * seq_len)
            contrastive_loss = F.cross_entropy(sim_flat, contrastive_labels)
            
            outputs['loss'] = lm_loss + 0.5 * cls_loss + 0.1 * contrastive_loss
            outputs['lm_loss'] = lm_loss
            outputs['cls_loss'] = cls_loss
            outputs['contrastive_loss'] = contrastive_loss
            
        return outputs


In [ ]:
# Data Loading - CVE Classification Dataset
# Using mounted Kaggle datasets

import pandas as pd
from torch.utils.data import Dataset, DataLoader

class CVEDataset(Dataset):
    def __init__(self, data_path, vocab_size=32000, max_len=256):
        self.max_len = max_len
        self.vocab_size = vocab_size
        
        # Try to load from Kaggle input
        possible_paths = [
            '/kaggle/input/cvefixes-vulnerable-and-fixed-code',
            '/kaggle/input/vulnerable-c-functions-for-security-prediction',
            '/kaggle/input/cve-fix-pairs',
            '/kaggle/input/vulnerable-programming-dataset',
            data_path
        ]
        
        self.data = None
        for path in possible_paths:
            if os.path.exists(path):
                print(f"Loading data from: {path}")
                self._load_from_path(path)
                break
        
        if self.data is None:
            print("No dataset found, generating synthetic CVE data")
            self._generate_synthetic()
    
    def _load_from_path(self, path):
        # Load JSON/CSV files from the dataset
        import glob
        files = glob.glob(os.path.join(path, '**/*'), recursive=True)
        for f in files:
            if f.endswith('.json'):
                try:
                    with open(f, 'r') as fp:
                        data = json.load(fp)
                    if isinstance(data, list) and len(data) > 0:
                        self.data = data
                        print(f"Loaded {len(data)} samples from {f}")
                        return
                except:
                    pass
            elif f.endswith('.csv'):
                try:
                    df = pd.read_csv(f)
                    if len(df) > 0:
                        self.data = df.to_dict('records')
                        print(f"Loaded {len(df)} samples from {f}")
                        return
                except:
                    pass
    
    def _generate_synthetic(self):
        # Generate synthetic CVE-like data for testing
        vuln_types = ['buffer_overflow', 'use_after_free', 'format_string', 'integer_overflow', 
                    'race_condition', 'sql_injection', 'xss', 'path_traversal', 
                    'command_injection', 'deserialization']
        
        templates = [
            "void vulnerable(char *input) {{ char buf[64]; strcpy(buf, input); }}",
            "void vulnerable(char *input) {{ char *ptr = malloc(10); free(ptr); *ptr = 'a'; }}",
            "void vulnerable(char *input) {{ printf(input); }}",
            "void vulnerable(int x) {{ int y = x + 1; if (y < x) {{ /* overflow */ }} }}",
            "void vulnerable() {{ /* race condition */ }}",
            "void vulnerable(char *input) {{ char query[100]; sprintf(query, \"SELECT * FROM t WHERE x=\"%s\"\", input); }}",
            "void vulnerable(char *input) {{ printf(\"<script>%s</script>\", input); }}",
            "void vulnerable(char *input) {{ char path[100]; sprintf(path, \"/var/www/%s\", input); }}",
            "void vulnerable(char *input) {{ system(input); }}",
            "void vulnerable(char *data) {{ deserialize(data); }}"
        ]
        
        self.data = []
        for i in range(10000):
            vuln_type = random.choice(vuln_types)
            template = random.choice(templates)
            self.data.append({
                'code': template,
                'label': vuln_types.index(vuln_type),
                'type': vuln_type
            })
        print(f"Generated {len(self.data)} synthetic samples")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        code = item.get('code', '')
        label = item.get('label', 0)
        
        # Simple character-level tokenization
        tokens = [ord(c) % (self.vocab_size - 100) + 100 for c in code[:self.max_len-2]]
        tokens = [1] + tokens + [2]  # BOS, EOS
        
        if len(tokens) < self.max_len:
            tokens += [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
        
        input_ids = torch.tensor(tokens, dtype=torch.long)
        targets = input_ids.clone()
        targets[:-1] = input_ids[1:]
        targets[-1] = 0
        
        return {
            'input_ids': input_ids,
            'targets': targets,
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Create datasets
train_dataset = CVEDataset('/kaggle/working/data', max_len=256)
val_size = min(1000, len(train_dataset) // 10)
train_size = len(train_dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")


In [ ]:
# Training Loop

model = NeuraleseReasoner(vocab_size=32000, latent_dim=1024, num_blocks=4, max_act_steps=8, num_classes=10).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

best_val_loss = float('inf')
best_val_acc = 0.0
patience = 10
patience_counter = 0

for epoch in range(100):
    # Training
    model.train()
    train_losses = []
    train_cls_acc = []
    
    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        targets = batch['targets'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        out = model(input_ids, targets=targets, labels=labels)
        loss = out['loss']
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_losses.append(loss.item())
        
        # Classification accuracy
        cls_preds = out['class_logits'].argmax(dim=-1)
        acc = (cls_preds == labels).float().mean().item()
        train_cls_acc.append(acc)
        
        if batch_idx % 50 == 0:
            print(f"Epoch {epoch} Batch {batch_idx}: Loss={loss.item():.4f}, Acc={acc:.4f}, Steps={out['avg_reasoning_steps']:.2f}")
    
    # Validation
    model.eval()
    val_losses = []
    val_cls_acc = []
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            targets = batch['targets'].to(device)
            labels = batch['labels'].to(device)
            
            out = model(input_ids, targets=targets, labels=labels)
            val_losses.append(out['loss'].item())
            
            cls_preds = out['class_logits'].argmax(dim=-1)
            acc = (cls_preds == labels).float().mean().item()
            val_cls_acc.append(acc)
    
    avg_train_loss = np.mean(train_losses)
    avg_train_acc = np.mean(train_cls_acc)
    avg_val_loss = np.mean(val_losses)
    avg_val_acc = np.mean(val_cls_acc)
    
    print(f"Epoch {epoch}: Train Loss={avg_train_loss:.4f}, Train Acc={avg_train_acc:.4f}, Val Loss={avg_val_loss:.4f}, Val Acc={avg_val_acc:.4f}")
    
    # Save best
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_val_acc = avg_val_acc
        patience_counter = 0
        
        # Save checkpoint
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': best_val_loss,
            'val_acc': best_val_acc,
            'config': {
                'vocab_size': 32000,
                'latent_dim': 1024,
                'num_blocks': 4,
                'max_act_steps': 8,
                'num_classes': 10
            }
        }, '/kaggle/working/neuralese_best.pt')
        print(f"  -> Saved best model (Val Loss: {best_val_loss:.4f}, Val Acc: {best_val_acc:.4f})")
    else:
        patience_counter += 1
        
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break
    
    scheduler.step()

print(f"Training complete. Best Val Loss: {best_val_loss:.4f}, Best Val Acc: {best_val_acc:.4f}")


In [ ]:
# Final Evaluation & Results Export

# Load best model
checkpoint = torch.load('/kaggle/working/neuralese_best.pt')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Test on validation set
test_losses = []
test_cls_acc = []
all_preds = []
all_labels = []
avg_steps = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        targets = batch['targets'].to(device)
        labels = batch['labels'].to(device)
        
        out = model(input_ids, targets=targets, labels=labels)
        test_losses.append(out['loss'].item())
        
        cls_preds = out['class_logits'].argmax(dim=-1)
        acc = (cls_preds == labels).float().mean().item()
        test_cls_acc.append(acc)
        
        all_preds.extend(cls_preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        avg_steps.append(out['avg_reasoning_steps'].item())

final_loss = np.mean(test_losses)
final_acc = np.mean(test_cls_acc)
avg_reasoning_steps = np.mean(avg_steps)

# Per-class accuracy
from sklearn.metrics import classification_report, confusion_matrix
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, digits=4))

print(f"\nFinal Results:")
print(f"  Test Loss: {final_loss:.4f}")
print(f"  Test Accuracy: {final_acc:.4f}")
print(f"  Avg Reasoning Steps: {avg_reasoning_steps:.2f}")
print(f"  Parameters: {total_params:,}")

# Export results for merge-chain
results = {
    'final_intel': final_acc,  # Using accuracy as proxy
    'final_loss': final_loss,
    'accuracy': final_acc,
    'avg_reasoning_steps': avg_reasoning_steps,
    'parameters': total_params,
    'config': checkpoint['config'],
    'epoch': checkpoint['epoch']
}

with open('/kaggle/working/cve_evo_kaggle_final.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\nResults saved to /kaggle/working/cve_evo_kaggle_final.json")
print(json.dumps(results, indent=2))
